This notebook checks whether normalized entropy can be used to choose low/medium/high certainty thresholds for the text-risk model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import csv
import math
import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# LIAR data is here
project_root = "/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1"
liar_data_dir = os.path.join(project_root, "data", "LIAR")
liar_data_valid = os.path.join(liar_data_dir, "valid.tsv")

# Model copy is here
model_project_root = "/content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/backend/fake_news_text_feature"
model_path = os.path.join(
    model_project_root,
    "model_training",
    "model_copy",
    "final_six_label_to_3risk_hf"
)

MAX_LENGTH = 128
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", project_root)
print("Model dir:", model_path)
print("Data file:", liar_data_valid)
print("Device:", DEVICE)

Project root: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1
Model dir: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/backend/fake_news_text_feature/model_training/model_copy/final_six_label_to_3risk_hf
Data file: /content/drive/MyDrive/UoP/COMP3000/fake-news-detector-v1/data/LIAR/valid.tsv
Device: cuda


In [3]:
LIAR_COLUMNS = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job",
    "state",
    "party",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context",
]

SIX_LABEL_TO_RISK_ID = {
    "pants-fire": 2,
    "false": 2,
    "barely-true": 1,
    "half-true": 1,
    "mostly-true": 0,
    "true": 0,
}

RISK_LABELS = ["low", "medium", "high"]


def load_liar_rows(data_file):
    rows = []

    with open(data_file, "r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file, fieldnames=LIAR_COLUMNS, delimiter="\t")
        for row in reader:
            if row["label"] in SIX_LABEL_TO_RISK_ID:
                rows.append(row)

    return rows


def make_batches(items, batch_size):
    for start_index in range(0, len(items), batch_size):
        yield items[start_index : start_index + batch_size]


rows = load_liar_rows(liar_data_valid)
print("Rows loaded:", len(rows))

Rows loaded: 1284


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(DEVICE)
model.eval()

print("Model loaded.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded.


In [5]:
results = []

with torch.no_grad():
    for batch in make_batches(rows, BATCH_SIZE):
        statements = [row["statement"] for row in batch]

        encoded = tokenizer(
            statements,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {name: value.to(DEVICE) for name, value in encoded.items()}

        logits = model(**encoded).logits
        six_label_probabilities = torch.softmax(logits, dim=1)

        low_probabilities = six_label_probabilities[:, 4] + six_label_probabilities[:, 5]
        medium_probabilities = six_label_probabilities[:, 2] + six_label_probabilities[:, 3]
        high_probabilities = six_label_probabilities[:, 0] + six_label_probabilities[:, 1]
        risk_probabilities = torch.stack(
            [low_probabilities, medium_probabilities, high_probabilities],
            dim=1,
        )

        predicted_risk_ids = torch.argmax(risk_probabilities, dim=1)
        top_probabilities = torch.max(risk_probabilities, dim=1).values

        entropy = -(risk_probabilities * torch.log(risk_probabilities.clamp_min(1e-12))).sum(dim=1)
        normalized_entropy = entropy / math.log(3)

        for row_index, row in enumerate(batch):
            true_risk_id = SIX_LABEL_TO_RISK_ID[row["label"]]
            predicted_risk_id = int(predicted_risk_ids[row_index].cpu().item())
            is_correct = predicted_risk_id == true_risk_id

            results.append(
                {
                    "statement": row["statement"],
                    "true_risk": RISK_LABELS[true_risk_id],
                    "predicted_risk": RISK_LABELS[predicted_risk_id],
                    "correct": is_correct,
                    "top_probability": float(top_probabilities[row_index].cpu().item()),
                    "normalized_entropy": float(normalized_entropy[row_index].cpu().item()),
                }
            )

accuracy = sum(row["correct"] for row in results) / len(results)
print("Finished predictions:", len(results))
print("Overall accuracy:", round(accuracy, 4))

Finished predictions: 1284
Overall accuracy: 0.493


In [6]:
def print_entropy_bins(results, bins):
    print("normalized_entropy_bin | count | accuracy | avg_top_probability")
    print("-" * 68)

    for bin_index in range(len(bins) - 1):
        left_edge = bins[bin_index]
        right_edge = bins[bin_index + 1]

        if bin_index == len(bins) - 2:
            rows_in_bin = [
                row for row in results
                if left_edge <= row["normalized_entropy"] <= right_edge
            ]
        else:
            rows_in_bin = [
                row for row in results
                if left_edge <= row["normalized_entropy"] < right_edge
            ]

        if rows_in_bin:
            bin_accuracy = sum(row["correct"] for row in rows_in_bin) / len(rows_in_bin)
            avg_top_probability = sum(row["top_probability"] for row in rows_in_bin) / len(rows_in_bin)
        else:
            bin_accuracy = 0
            avg_top_probability = 0

        print(
            f"{left_edge:.2f}-{right_edge:.2f}".ljust(23),
            str(len(rows_in_bin)).rjust(5),
            f"{bin_accuracy:.4f}".rjust(10),
            f"{avg_top_probability:.4f}".rjust(20),
        )


entropy_bins = [0.0, 0.5, 0.65, 0.75, 0.85, 0.95, 1.0]
print_entropy_bins(results, entropy_bins)

normalized_entropy_bin | count | accuracy | avg_top_probability
--------------------------------------------------------------------
0.00-0.50                  83     0.6747               0.9051
0.50-0.65                 135     0.5852               0.7893
0.65-0.75                 176     0.5682               0.7012
0.75-0.85                 300     0.4767               0.6060
0.85-0.95                 399     0.4511               0.5168
0.95-1.00                 191     0.3927               0.4217


In [7]:
def get_certainty_level(normalized_entropy, low_cutoff=0.85, high_cutoff=0.65):
    if normalized_entropy >= low_cutoff:
        return "low"
    if normalized_entropy >= high_cutoff:
        return "medium"
    return "high"


def print_certainty_groups(results, low_cutoff=0.85, high_cutoff=0.65):
    groups = {
        "low": [],
        "medium": [],
        "high": [],
    }

    for row in results:
        certainty_level = get_certainty_level(
            row["normalized_entropy"],
            low_cutoff=low_cutoff,
            high_cutoff=high_cutoff,
        )
        groups[certainty_level].append(row)

    print("certainty_level | count | accuracy | avg_normalized_entropy")
    print("-" * 70)

    for certainty_level in ["low", "medium", "high"]:
        group_rows = groups[certainty_level]

        if group_rows:
            group_accuracy = sum(row["correct"] for row in group_rows) / len(group_rows)
            avg_entropy = sum(row["normalized_entropy"] for row in group_rows) / len(group_rows)
        else:
            group_accuracy = 0
            avg_entropy = 0

        print(
            certainty_level.ljust(15),
            str(len(group_rows)).rjust(5),
            f"{group_accuracy:.4f}".rjust(10),
            f"{avg_entropy:.4f}".rjust(25),
        )


LOW_CERTAINTY_CUTOFF = 0.85
HIGH_CERTAINTY_CUTOFF = 0.65

print("Current cutoffs:")
print("High certainty if normalized_entropy <", HIGH_CERTAINTY_CUTOFF)
print("Low certainty if normalized_entropy >=", LOW_CERTAINTY_CUTOFF)
print()
print_certainty_groups(
    results,
    low_cutoff=LOW_CERTAINTY_CUTOFF,
    high_cutoff=HIGH_CERTAINTY_CUTOFF,
)

Current cutoffs:
High certainty if normalized_entropy < 0.65
Low certainty if normalized_entropy >= 0.85

certainty_level | count | accuracy | avg_normalized_entropy
----------------------------------------------------------------------
low               590     0.4322                    0.9226
medium            476     0.5105                    0.7672
high              218     0.6193                    0.4833


In [8]:
print("Lowest entropy examples")
for row in sorted(results, key=lambda item: item["normalized_entropy"])[:5]:
    print("-", round(row["normalized_entropy"], 4), row["predicted_risk"], "correct=", row["correct"], "|", row["statement"][:140])

print()
print("Highest entropy examples")
for row in sorted(results, key=lambda item: item["normalized_entropy"], reverse=True)[:5]:
    print("-", round(row["normalized_entropy"], 4), row["predicted_risk"], "correct=", row["correct"], "|", row["statement"][:140])

Lowest entropy examples
- 0.083 high correct= True | There was serious voter fraud in California.
- 0.086 high correct= True | Says Barack Obama is a Muslim.
- 0.0869 high correct= True | A photo from the Russia-Finland hockey game shows a sad Vladimir Putin and Dmitry Medvedev.
- 0.0885 high correct= True | A hidden provision in Obamacare taxes sporting goods as medical devices.
- 0.0943 high correct= True | Bernie Sanders admits he is a Democratic Socialist. Nazis were Democratic Socialists.

Highest entropy examples
- 0.9996 medium correct= True | Says that under Gov. Rick Perry, Texas Department of Public Safety troopers have had standing orders not to inquire into the immigration sta
- 0.999 medium correct= False | Says he did not say Republicans would filibuster immigration reform.
- 0.9989 medium correct= True | The IRS scandal clearly showed some criminal behavior.
- 0.9981 medium correct= False | For the first time in history, the Democratic Congress will not allow an increase